# Local RAG Evaluation Workbook

Notebook này là bản self-contained: vừa giải thích framework/metric, vừa chứa code evaluator, vừa chạy benchmark và đọc report. Không cần file `rag_eval_local.py` riêng nữa.

Thư mục `local_eval/` đang được ignore bằng `.git/info/exclude`, nên notebook này chỉ dùng local và không publish lên GitHub.

## Cách hiểu đúng

Đây không phải fine-tune model.

- **Fine-tune**: huấn luyện/chỉnh trọng số model.
- **Custom metric**: tự thiết kế thước đo phù hợp domain.

Cách làm ở project này:

```text
Metric backbone chuẩn của Information Retrieval
+ Metric custom cho pháp luật Việt Nam
= Domain-adapted RAG evaluation
```

## Paper/framework được dùng làm khung tư duy

| Paper/framework | Lấy ý tưởng gì | Áp vào chatbot luật Việt Nam | Trong notebook/code |
|---|---|---|---|
| RAGVUE | Tách lỗi RAG thành retrieval, answer quality, grounding, diagnostic | Biết lỗi nằm ở retrieval hay generation/grounding | `retrieval_metrics()`, `answer_metrics()`, `recommendations()` |
| Case-Aware | Đánh giá theo case, identifier integrity, severity-aware scoring | Mỗi câu hỏi là một legal case; đúng số hiệu, đúng điều, đúng mẫu | `DEFAULT_CASES`, `identifier_integrity_score`, `SEVERITY_WEIGHTS` |
| RARE | Redundancy-aware retrieval cho corpus giống nhau cao | Luật, nghị định, thông tư có nhiều đoạn tương tự nhau | `redundancy_metrics()`, `dominant_law_share`, `near_duplicate_rate` |
| RAGe | Đo latency/resource local | Biết app chạy trên laptop mất bao lâu, tốn memory bao nhiêu | `retrieval_ms`, `generation_ms`, `peak_tracemalloc_mb` |
| StepGap | Phát hiện thiếu bằng chứng và gợi ý repair | Nếu sai điều/citation/identifier thì đưa hướng sửa cụ thể | `unsupported_identifiers`, `grounding_score`, `recommendations()` |

## Metric chuẩn và metric custom

### Retrieval backbone metrics

| Metric | Dịch nghĩa | Loại | Ý nghĩa |
|---|---|---|---|
| `retrieval_recall_at_k` | Độ bao phủ truy xuất trong top-k | Backbone chuẩn | Top-k có chứa chunk/document kỳ vọng không |
| `retrieval_mrr` | Hạng nghịch đảo trung bình | Backbone chuẩn | Chunk đúng càng gần top 1 thì điểm càng cao |
| `retrieval_ndcg_at_k` | Chất lượng xếp hạng giảm trọng số theo vị trí | Backbone chuẩn | Ranking các chunk liên quan có tốt không |

### Legal-specific custom metrics

| Metric | Dịch nghĩa | Loại | Ý nghĩa |
|---|---|---|---|
| `identifier_integrity_score` | Điểm toàn vẹn định danh pháp lý | Custom domain luật | Có giữ đúng số hiệu, điều, Mẫu số, Phụ lục không |
| `citation_validity` | Độ hợp lệ citation | Custom cho RAG UI | Citation `[1]`, `[2]` có trỏ tới nguồn retrieved thật không |
| `answer_required_term_score` | Điểm độ ý bắt buộc | Custom checklist luật | Câu trả lời có đủ ý pháp lý quan trọng không |
| `unsupported_identifiers` | Định danh không có trong bằng chứng | Custom grounding luật | Model có bịa điều/Mẫu/Phụ lục không nằm trong context không |
| `severity_weighted_score` | Điểm có trọng số theo mức nghiêm trọng | Custom theo case | Lỗi ở case quan trọng bị phạt nặng hơn |

## Evaluator code

Cell dưới nạp evaluator dùng cùng `guarded_retrieve()` và fallback với chatbot thật. Logic chính nằm tại `local_eval/rag_eval_local.py`.


In [ ]:
from local_eval.rag_eval_local import *  # noqa: F401,F403

print("Loaded guard-aware evaluator")
print(f"Cases: {len(DEFAULT_CASES)}")


## Chạy benchmark guard + retrieval

Chế độ mặc định không cần Ollama. Nó đo quyết định accept/reject, false acceptance/rejection, fallback, rewrite, retrieval ranking và latency.


In [ ]:
# Đảm bảo Cell 5 (Evaluator code) đã được chạy trước khi chạy cell này.
try:
    DEFAULT_QWEN_MODEL
except NameError:
    raise RuntimeError("Hãy chạy Cell 'Evaluator code' trước!")

import datetime
import subprocess
from types import SimpleNamespace

# ════════════════════════════════════════════════════════════════════════════
# ✏️  Chỉ cần thay đổi 1 dòng này nếu muốn ghi chú — để None thì tự lấy từ git
EXP_NOTE_OVERRIDE = None   # ví dụ: "thêm reranker bge" — hoặc để None
GENERATE          = False  # True = cần Ollama đang chạy
# ════════════════════════════════════════════════════════════════════════════

# ── Auto-detect context từ git ───────────────────────────────────────────────
def _git(cmd):
    try:
        return subprocess.check_output(
            cmd, cwd=str(PROJECT_ROOT), stderr=subprocess.DEVNULL, text=True
        ).strip()
    except Exception:
        return ""

_commit_hash    = _git(["git", "log", "-1", "--format=%h"])
_commit_msg     = _git(["git", "log", "-1", "--format=%s"])
_commit_author  = _git(["git", "log", "-1", "--format=%an"])
_commit_time    = _git(["git", "log", "-1", "--format=%ci"])
_branch         = _git(["git", "rev-parse", "--abbrev-ref", "HEAD"])

# Diff so với commit trước (chỉ tên file thay đổi)
_changed_files  = _git(["git", "diff", "HEAD~1", "--name-only"])
# Staged nhưng chưa commit
_staged_files   = _git(["git", "diff", "--cached", "--name-only"])
# Unstaged (đang sửa dở)
_dirty_files    = _git(["git", "diff", "--name-only"])

# Snapshot nội dung diff ngắn gọn (tối đa 4000 chars) để lưu vào log
_diff_stat      = _git(["git", "diff", "HEAD~1", "--stat"])
_diff_patch     = _git(["git", "diff", "HEAD~1", "--", 
                         "step3_rag_chatbot.py",
                         "step1_chunking.py", "step2_vectorstore.py",
                         "chunking.py", "vectorstore.py",
                         "retrieval.py", "reranker.py"])[:4000]

# Auto note: git commit msg > override > fallback
_auto_note = EXP_NOTE_OVERRIDE or _commit_msg or "no-git"
if _dirty_files or _staged_files:
    _auto_note += " [+uncommitted]"

# ── Run evaluation ───────────────────────────────────────────────────────────
args = SimpleNamespace(
    vectorstore="vectorstore/legal_vectorstore.json",
    cases=None,
    top_k=DEFAULT_TOP_K,
    generate=GENERATE,
    model=os.getenv("QWEN_MODEL", DEFAULT_QWEN_MODEL),
    ollama_url=os.getenv("OLLAMA_URL", "http://localhost:11434"),
    temperature=0.0,
    max_context_chars=12000,
    out_dir="local_eval/reports",
)

cases = load_cases(args.cases)
manifest, documents, vectors = load_vectorstore(PROJECT_ROOT / args.vectorstore)
model = load_embedding_model(manifest["embedding_model"])
runtime = {"manifest": manifest, "documents": documents, "vectors": vectors, "model": model}

results = [evaluate_case(case, runtime, args) for case in cases]
summary = summarize(results, args.generate)
write_reports(results, summary, PROJECT_ROOT / args.out_dir)

# ── Ghi experiment log ───────────────────────────────────────────────────────
_exp_record = {
    "timestamp":   datetime.datetime.now().isoformat(timespec="seconds"),
    "note":        _auto_note,
    "git": {
        "branch":         _branch,
        "commit_hash":    _commit_hash,
        "commit_msg":     _commit_msg,
        "commit_author":  _commit_author,
        "commit_time":    _commit_time,
        "changed_files":  _changed_files.splitlines() if _changed_files else [],
        "staged_files":   _staged_files.splitlines() if _staged_files else [],
        "dirty_files":    _dirty_files.splitlines() if _dirty_files else [],
        "diff_stat":      _diff_stat,
        "diff_patch":     _diff_patch,
    },
    "runtime": {
        "top_k":           args.top_k,
        "embedding_model": manifest["embedding_model"],
        "llm_model":       args.model,
        "generate":        GENERATE,
        "max_context_chars": args.max_context_chars,
    },
    "summary": summary,
}

_exp_log_path = PROJECT_ROOT / "local_eval" / "experiments.jsonl"
_exp_log_path.parent.mkdir(parents=True, exist_ok=True)
with _exp_log_path.open("a", encoding="utf-8") as _f:
    _f.write(json.dumps(_exp_record, ensure_ascii=False) + "\n")

# ── In tóm tắt nhanh ─────────────────────────────────────────────────────────
print(f"✅  Run logged: \"{_auto_note}\"")
print(f"    branch={_branch}  commit={_commit_hash}  ({_commit_msg})")
if _dirty_files:
    print(f"    ⚠️  File đang sửa dở (chưa commit): {_dirty_files.replace(chr(10), ', ')}")
if _staged_files:
    print(f"    📌 File staged: {_staged_files.replace(chr(10), ', ')}")
print(f"    📁 Log → {_exp_log_path}")
summary

## Summary report

In [ ]:
report = json.loads((PROJECT_ROOT / "local_eval/reports/rag_eval_report.json").read_text(encoding="utf-8"))
summary = report["summary"]

try:
    from IPython.display import Markdown, display
    rows = ["| Metric | Value |", "|---|---:|"]
    rows += [f"| `{key}` | {value} |" for key, value in summary.items()]
    display(Markdown("\n".join(rows)))
except Exception:
    print(json.dumps(summary, ensure_ascii=False, indent=2))

## Case-level report

In [ ]:
import csv

with (PROJECT_ROOT / "local_eval/reports/rag_eval_cases.csv").open("r", encoding="utf-8-sig", newline="") as handle:
    case_rows = list(csv.DictReader(handle))

try:
    from IPython.display import Markdown, display
    rows = [
        "| Case | Score | Recall@K | MRR | nDCG@K | Top source | Recommendation |",
        "|---|---:|---:|---:|---:|---|---|",
    ]
    for row in case_rows:
        top_source = row["top_source"].replace("|", "/")
        rec = row["recommendations"].replace("|", "<br>")
        rows.append(
            f"| `{row['id']}` | {row['case_score']} | {row['recall_at_k']} | {row['mrr']} | {row['ndcg_at_k']} | {top_source} | {rec} |"
        )
    display(Markdown("\n".join(rows)))
except Exception:
    for row in case_rows:
        print(row)

## Roadmap cải thiện theo metric

| Nếu metric xấu | Ý nghĩa | Sửa ở đâu |
|---|---|---|
| `guard_decision_accuracy` thấp | Phân loại câu hợp lệ/câu xấu sai | `analyze_question()` |
| `false_rejection_rate` cao | Câu pháp lý đúng bị chặn | domain signals hoặc threshold |
| `false_acceptance_rate` cao | Câu ngoài phạm vi vẫn lấy chunk | query guard và score/gap |
| `fallback_accuracy` thấp | Câu bị từ chối vẫn gọi LLM/trả source | tầng Python/API |
| `rewrite_accuracy` thấp | Rewrite làm sai hoặc mất ý | `rewrite_query()` |
| `retrieval_recall_at_k` thấp | Không lấy được chunk đúng | embedding, boost, reranker |
| `citation_validity` thấp | Câu trả lời thiếu citation hợp lệ | generation retry/post-check |


## Optional: chạy full evaluation với Qwen/Ollama

Chỉ chạy khi Ollama đang hoạt động ở `http://localhost:11434`. Khi bật `generate=True`, evaluator sẽ có thêm `grounding_score`, `citation_validity`, `identifier_integrity_score`, `answer_required_term_score`.

In [ ]:
# Chỉ chạy cell này khi Ollama đang chạy ở http://localhost:11434.
# args.generate = True
# results = [evaluate_case(case, runtime, args) for case in cases]
# summary = summarize(results, args.generate)
# write_reports(results, summary, PROJECT_ROOT / args.out_dir)
# summary

## So sánh các lần thử nghiệm

Mỗi lần chạy Cell "Benchmark" ở trên sẽ tự động lưu config + metric vào `local_eval/experiments.jsonl`.  
Cell dưới đọc toàn bộ lịch sử và highlight **xanh** (tốt hơn) / **đỏ** (xấu hơn) so với lần trước.

In [ ]:
# ── So sánh tất cả experiment runs ─────────────────────────────────────────
_exp_log_path = PROJECT_ROOT / "local_eval" / "experiments.jsonl"

if not _exp_log_path.exists():
    print("Chưa có experiment nào. Chạy Cell benchmark trước.")
else:
    _exps = []
    with _exp_log_path.open("r", encoding="utf-8") as _f:
        for _line in _f:
            _line = _line.strip()
            if _line:
                _exps.append(json.loads(_line))

    if len(_exps) == 0:
        print("File experiments.jsonl trống.")
    else:
        _METRICS = [
            # (key,                          label,              higher_is_better)
            ("overall_score",               "Overall",          True),
            ("severity_weighted_score",     "Sev-W Score",      True),
            ("retrieval_recall_at_k",       "Recall@K",         True),
            ("retrieval_mrr",               "MRR",              True),
            ("retrieval_ndcg_at_k",         "nDCG@K",           True),
            ("near_duplicate_rate",         "NearDup↓",         False),
            ("dominant_law_share",          "DomLaw↓",          False),
            ("retrieval_ms_avg",            "Retr ms↓",         False),
            ("grounding_score",             "Grounding",        True),
            ("identifier_integrity_score",  "Identifier",       True),
            ("answer_required_term_score",  "ReqTerm",          True),
            ("citation_validity",           "Citation",         True),
        ]

        # Lấy baseline = lần chạy đầu tiên
        _baseline_summary = _exps[0]["summary"]

        try:
            from IPython.display import HTML, display

            def _cell(val, baseline_val, prev_val, hb):
                """Render một ô: màu so vs baseline, mũi tên so vs lần trước."""
                if val is None:
                    return "<td>—</td>"
                if not isinstance(val, (int, float)):
                    return f"<td>{val}</td>"

                # Màu nền so với BASELINE
                if baseline_val is not None and val != baseline_val:
                    better_than_base = (val > baseline_val) if hb else (val < baseline_val)
                    bg = "#c8f7c5" if better_than_base else "#f7c5c5"
                else:
                    bg = "transparent"

                # Mũi tên so với LẦN TRƯỚC
                arrow = ""
                if prev_val is not None and val != prev_val:
                    better_than_prev = (val > prev_val) if hb else (val < prev_val)
                    arrow = " <span style=\'color:#1a7a1a\'>▲</span>" if better_than_prev \
                            else " <span style=\'color:#a01010\'>▼</span>"

                return f'<td style="background:{bg};text-align:right">{val:.4f}{arrow}</td>'

            rows = []
            # Header
            rows.append(
                "<tr>"
                "<th>#</th><th>Timestamp</th><th>Note (git commit)</th>"
                "<th>Branch</th><th>Commit</th><th>Changed files</th>"
                + "".join(f"<th>{lbl}</th>" for _, lbl, _ in _METRICS)
                + "</tr>"
            )

            _prev_s = None
            for _i, _exp in enumerate(_exps):
                _s   = _exp["summary"]
                _git = _exp.get("git", {})
                _is_baseline = (_i == 0)

                _changed = "<br>".join(_git.get("changed_files", [])) or "—"
                _dirty   = _git.get("dirty_files", [])
                _note_td = f"<b>{_exp['note']}</b>"
                if _dirty:
                    _note_td += f" <span style=\'color:#e67e22\' title=\'{', '.join(_dirty)}\'>⚠️</span>"

                _cells = [
                    f"<td><b>{'★' if _is_baseline else '#' + str(_i+1)}</b></td>",
                    f"<td style=\'white-space:nowrap\'>{_exp['timestamp']}</td>",
                    f"<td>{_note_td}</td>",
                    f"<td>{_git.get('branch','—')}</td>",
                    f"<td style=\'font-family:monospace\'>{_git.get('commit_hash','—')}</td>",
                    f"<td style=\'font-size:11px;color:#555\'>{_changed}</td>",
                ]

                for _key, _, _hb in _METRICS:
                    _val  = _s.get(_key)
                    _base = _baseline_summary.get(_key)
                    _prev = _prev_s.get(_key) if _prev_s else None
                    if _is_baseline:
                        # Baseline row: no color, just value
                        _cells.append(
                            f"<td style=\'text-align:right\'>"
                            f"{'—' if _val is None else f'{_val:.4f}'}</td>"
                        )
                    else:
                        _cells.append(_cell(_val, _base, _prev, _hb))

                rows.append(f"<tr>{''.join(_cells)}</tr>")
                _prev_s = _s

            _style = """<style>
            .exp-tbl{border-collapse:collapse;font-size:12px;width:100%}
            .exp-tbl th,.exp-tbl td{border:1px solid #ccc;padding:4px 8px;white-space:nowrap}
            .exp-tbl th{background:#2c3e50;color:#fff;text-align:center;position:sticky;top:0}
            .exp-tbl tr:first-child td,.exp-tbl tr:first-child th{font-weight:bold}
            </style>"""
            _intro = (
                "<p><b>★ baseline</b> = lần chạy đầu tiên. "
                "Màu xanh/đỏ = so với baseline. Mũi tên ▲▼ = so với lần trước.</p>"
            )
            _table = '<div style="overflow-x:auto"><table class="exp-tbl">' + "".join(rows) + "</table></div>"
            _footer = f'<p style="font-size:11px;color:#888">📁 {_exp_log_path} — {len(_exps)} run(s)</p>'
            _html = _style + _intro + _table + _footer
            display(HTML(_html))

        except Exception as _e:
            # Plain-text fallback
            _keys = [k for k, _, _ in _METRICS]
            print("\t".join(["#", "timestamp", "note", "branch", "commit"] + _keys))
            for _i, _exp in enumerate(_exps):
                _s = _exp["summary"]; _g = _exp.get("git", {})
                print("\t".join(
                    [str(_i+1), _exp["timestamp"], _exp["note"],
                     _g.get("branch",""), _g.get("commit_hash","")]
                    + [str(_s.get(k,"")) for k in _keys]
                ))

## Delta chart — thay đổi metric qua các lần chạy

Chart dưới vẽ **sự thay đổi** (Δ) của từng metric so với lần chạy đầu tiên (baseline).  
Cột xanh = tốt hơn baseline, cột đỏ = xấu hơn.

In [ ]:
# ── Delta chart: mỗi run so với baseline ────────────────────────────────────
_exp_log_path = PROJECT_ROOT / "local_eval" / "experiments.jsonl"

if not _exp_log_path.exists():
    print("Chưa có experiment nào.")
else:
    _exps = []
    with _exp_log_path.open("r", encoding="utf-8") as _f:
        for _l in _f:
            _l = _l.strip()
            if _l: _exps.append(json.loads(_l))

    if len(_exps) < 2:
        print("Cần ít nhất 2 lần chạy để vẽ delta chart.")
    else:
        _PLOT = [
            ("overall_score",               "Overall",     True),
            ("severity_weighted_score",     "Sev-W",       True),
            ("retrieval_recall_at_k",       "Recall@K",    True),
            ("retrieval_mrr",               "MRR",         True),
            ("retrieval_ndcg_at_k",         "nDCG@K",      True),
            ("near_duplicate_rate",         "NearDup",     False),
            ("dominant_law_share",          "DomLaw",      False),
            ("retrieval_ms_avg",            "Retr ms",     False),
            ("grounding_score",             "Grounding",   True),
            ("identifier_integrity_score",  "Identifier",  True),
        ]
        _base = _exps[0]["summary"]

        try:
            import matplotlib; matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            import numpy as np, io
            from IPython.display import Image, display as _disp

            _n_runs = len(_exps) - 1   # runs after baseline
            _n_met  = len(_PLOT)
            _x      = np.arange(_n_met)
            _w      = min(0.7 / max(_n_runs, 1), 0.35)

            fig, ax = plt.subplots(figsize=(max(12, _n_met * 1.5), 5))
            ax.axhline(0, color="black", lw=1, ls="--", alpha=0.5)

            for _i, _exp in enumerate(_exps[1:]):
                _s = _exp["summary"]
                _g = _exp.get("git", {})
                _label = f"#{_i+2} {_g.get('commit_hash','')[:6]} {_exp['note'][:20]}"
                _dirty = "⚠️ " if _g.get("dirty_files") else ""

                _deltas, _colors = [], []
                for _key, _, _hb in _PLOT:
                    _v = _s.get(_key); _b = _base.get(_key)
                    if _v is None or _b is None:
                        _deltas.append(0.0)
                    else:
                        _d = (_v - _b) if _hb else (_b - _v)  # flip: lower-is-better
                        _deltas.append(_d)
                    _colors.append("#2ecc71" if _deltas[-1] >= 0 else "#e74c3c")

                _offset = (_i - (_n_runs - 1) / 2) * _w
                _bars = ax.bar(_x + _offset, _deltas, _w * 0.92,
                               color=_colors, label=f"{_dirty}{_label}", alpha=0.85)
                for _bar, _d in zip(_bars, _deltas):
                    if abs(_d) > 0.003:
                        ax.text(_bar.get_x() + _bar.get_width() / 2, _d,
                                f"{_d:+.3f}", ha="center",
                                va="bottom" if _d >= 0 else "top", fontsize=7)

            ax.set_xticks(_x)
            ax.set_xticklabels([lbl for _, lbl, _ in _PLOT], fontsize=9)
            ax.set_ylabel("Δ vs baseline  (lower-is-better metrics đã flip)")
            ax.set_title("Metric Δ vs baseline — xanh = cải thiện, đỏ = xấu hơn")
            ax.legend(fontsize=8, loc="upper right")
            ax.grid(axis="y", alpha=0.25)
            plt.tight_layout()

            _buf = io.BytesIO()
            plt.savefig(_buf, format="png", dpi=130)
            _out_png = PROJECT_ROOT / "local_eval" / "reports" / "delta_chart.png"
            plt.savefig(str(_out_png), dpi=130)
            plt.close(); _buf.seek(0)
            _disp(Image(data=_buf.read()))
            print(f"Chart đã lưu: {_out_png}")

        except ImportError:
            # ASCII fallback
            print("matplotlib không có — ASCII fallback:\n")
            for _i, _exp in enumerate(_exps[1:]):
                _s = _exp["summary"]; _g = _exp.get("git", {})
                print(f"--- #{_i+2} {_g.get('commit_hash','')[:7]} {_exp['note']} ---")
                _changed = ", ".join(_g.get("changed_files", ["-"]))
                print(f"    changed: {_changed}")
                for _key, _lbl, _hb in _PLOT:
                    _v = _s.get(_key); _b = _base.get(_key)
                    if _v is None or _b is None: continue
                    _d = (_v - _b) if _hb else (_b - _v)
                    _sym = "▲" if _d > 0 else ("▼" if _d < 0 else "=")
                    print(f"    {_lbl:<22}: {_v:.4f}  {_sym} {_d:+.4f}  (baseline {_b:.4f})")

## Xem git diff của từng lần chạy

Cell dưới cho phép xem chính xác bạn đã **thay đổi gì trong code** ở mỗi lần experiment — kết nối trực tiếp metric thay đổi với dòng code đã sửa.

In [ ]:
# ── Xem diff chi tiết của một run cụ thể ────────────────────────────────────
_exp_log_path = PROJECT_ROOT / "local_eval" / "experiments.jsonl"

# ✏️ Đổi số này để xem run nào (1 = baseline, 2 = run thứ 2, ...)
VIEW_RUN = 2

if not _exp_log_path.exists():
    print("Chưa có experiment nào.")
else:
    _exps = []
    with _exp_log_path.open("r", encoding="utf-8") as _f:
        for _l in _f:
            _l = _l.strip()
            if _l: _exps.append(json.loads(_l))

    if VIEW_RUN < 1 or VIEW_RUN > len(_exps):
        print(f"Không có run #{VIEW_RUN}. Hiện có {len(_exps)} run(s).")
    else:
        _exp = _exps[VIEW_RUN - 1]
        _g   = _exp.get("git", {})
        _s   = _exp["summary"]
        _base_s = _exps[0]["summary"]

        print(f"{'='*60}")
        print(f"Run #{VIEW_RUN}: {_exp['note']}")
        print(f"Timestamp : {_exp['timestamp']}")
        print(f"Branch    : {_g.get('branch','—')}")
        print(f"Commit    : {_g.get('commit_hash','—')} — {_g.get('commit_msg','—')}")
        print(f"Author    : {_g.get('commit_author','—')}  {_g.get('commit_time','—')}")
        print()

        _changed = _g.get("changed_files", [])
        _dirty   = _g.get("dirty_files", [])
        _staged  = _g.get("staged_files", [])
        if _changed: print("Files thay đổi (vs commit trước):\n  " + "\n  ".join(_changed))
        if _staged:  print("Files staged (chưa commit):\n  " + "\n  ".join(_staged))
        if _dirty:   print("⚠️  Files đang sửa dở (unstaged):\n  " + "\n  ".join(_dirty))

        print()
        print("── Diff stat ──────────────────────────────────────")
        print(_g.get("diff_stat") or "(không có diff stat)")

        print()
        print("── Metric so với baseline ─────────────────────────")
        _SHOW = [
            ("overall_score",              "Overall Score",           True),
            ("severity_weighted_score",    "Severity-Weighted Score", True),
            ("retrieval_recall_at_k",      "Recall@K",                True),
            ("retrieval_mrr",              "MRR",                     True),
            ("retrieval_ndcg_at_k",        "nDCG@K",                  True),
            ("near_duplicate_rate",        "Near-Dup Rate",           False),
            ("grounding_score",            "Grounding Score",         True),
            ("identifier_integrity_score", "Identifier Integrity",    True),
            ("citation_validity",          "Citation Validity",       True),
            ("retrieval_ms_avg",           "Retrieval ms",            False),
        ]
        if VIEW_RUN == 1:
            print("(Đây là baseline — không có delta)")
        for _key, _lbl, _hb in _SHOW:
            _v = _s.get(_key); _b = _base_s.get(_key)
            if _v is None: continue
            if VIEW_RUN == 1 or _b is None:
                print(f"  {_lbl:<28}: {_v:.4f}")
            else:
                _d = (_v - _b) if _hb else (_b - _v)
                _sym = ("✅ ▲" if _d > 0 else ("❌ ▼" if _d < 0 else "  ="))
                print(f"  {_lbl:<28}: {_v:.4f}  {_sym} {_d:+.4f}  (baseline {_b:.4f})")

        if VIEW_RUN > 1:
            print()
            print("── Diff patch (key files) ─────────────────────────")
            _patch = _g.get("diff_patch", "")
            print(_patch if _patch else "(không có patch — có thể chưa commit thay đổi)")

## Cách nói với nhà tuyển dụng

> Em không chỉ dùng chatbot RAG, mà còn xây một local evaluation workbook. Em dùng các metric IR chuẩn như Recall@K, MRR, nDCG làm retrieval backbone, sau đó custom thêm legal-specific metrics như identifier integrity, citation validity, required legal terms và severity-weighted score để đánh giá đúng đặc thù pháp luật Việt Nam. Từ report, em biết lỗi nằm ở retrieval, grounding hay generation để can thiệp đúng phần code.